<a href="https://colab.research.google.com/github/fuzzynetwork1989-alt/synova-workspace/blob/main/ml/training/finetune_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synova AI Fine-Tuning Pipeline
## Fine-tune Llama 3.1 8B on Synova-specific dataset using free GPU resources

## Setup
Install required libraries

In [1]:
!pip install -q torch transformers peft datasets bitsandbytes trl accelerate
!pip install -q wandb  # For experiment tracking (optional)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.7 MB/s eta 0:00:00


## Import Libraries

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import json
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Configuration

In [ ]:
# Model configuration
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B"  # Base model
OUTPUT_DIR = "./synova-finetuned"

# Training configuration
BATCH_SIZE = 4  # Adjust based on GPU memory
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512

# LoRA configuration (parameter-efficient fine-tuning)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

print("Configuration loaded")

## Load Dataset

In [ ]:
# Option 1: Upload your dataset to Colab
# from google.colab import files
# uploaded = files.upload()
# dataset_path = list(uploaded.keys())[0]

# Option 2: Load from Hugging Face (if uploaded)
# dataset = load_dataset("your-username/synova-dataset")

# Option 3: Create sample dataset for testing
# Replace this with your actual dataset loading
sample_data = [
    {
        "instruction": "How should I implement JWT authentication in a Synova API endpoint?",
        "input": "",
        "output": "In Synova, JWT authentication is implemented using the middleware/auth.py module. Use the get_current_user dependency in your endpoint, and include the Authorization: Bearer <token> header in requests. The token is generated using the generate_token.py script."
    },
    {
        "instruction": "What is the recommended way to structure a new API router in Synova?",
        "input": "",
        "output": "Create a new file in apps/api/src/routers/ with an APIRouter instance. Define your endpoints with proper typing, error handling, and logging. Register the router in main.py with app.include_router(). Follow the patterns in existing routers like chat.py or agents.py."
    }
]

# For actual use, load your JSONL file
# def load_jsonl(file_path):
#     data = []
#     with open(file_path, 'r') as f:
#         for line in f:
#             data.append(json.loads(line))
#     return data

# dataset = load_jsonl("synova_dataset.jsonl")

print(f"Loaded {len(sample_data)} examples")
print(f"\nSample example:")
print(json.dumps(sample_data[0], indent=2))

## Format Dataset for Training

In [ ]:
def format_example(example):
    """Format example for instruction tuning"""
    instruction = example["instruction"]
    input_text = example.get("input", "")
    output = example["output"]

    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"

    return prompt

# Format all examples
formatted_data = [format_example(ex) for ex in sample_data]

print(f"Formatted {len(formatted_data)} examples")
print(f"\nSample formatted example:")
print(formatted_data[0][:500] + "...")

## Load Model and Tokenizer

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model in 4-bit quantization (for memory efficiency)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)

print(f"Model loaded with LoRA")
print(f"Trainable parameters: {model.print_trainable_parameters()}")

## Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length"
    )

# Tokenize
tokenized_dataset = [tokenize_function(text) for text in formatted_data]

# Convert to dataset format
from datasets import Dataset
train_dataset = Dataset.from_list(tokenized_dataset)

# Split into train/validation
train_dataset = train_dataset.train_test_split(test_size=0.1)

print(f"Train set size: {len(train_dataset['train'])}")
print(f"Validation set size: {len(train_dataset['test'])}")

## Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=100,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",  # Set to "wandb" if using Weights & Biases
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

print("Training arguments configured")

## Initialize Trainer

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal language modeling
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset["train"],
    eval_dataset=train_dataset["test"],
    data_collator=data_collator,
)

print("Trainer initialized")

## Start Training

In [ ]:
print("Starting training...")
trainer.train()
print("Training complete!")

## Save Model

In [ ]:
# Save the fine-tuned model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to {OUTPUT_DIR}")

## Test the Fine-Tuned Model

In [ ]:
# Load the fine-tuned model
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

# Test with a Synova-specific question
test_instruction = "How do I add a new endpoint to the Synova API?"
test_prompt = f"### Instruction:\n{test_instruction}\n\n### Response:\n"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Test response:")
print(response)

## Download Model (for local use)

In [ ]:
# Zip the model directory
!zip -r synova-finetuned.zip synova-finetuned/

# Download
from google.colab import files
files.download("synova-finetuned.zip")

## Optional: Upload to Hugging Face
```python
from huggingface_hub import login
login(token="your_hf_token")

model.push_to_hub("your-username/synova-llama-3.1-8b")
tokenizer.push_to_hub("your-username/synova-llama-3.1-8b")
```